In [65]:
import pandas as pd

# Load Google data
google_data = pd.read_csv(
    "/home/mbiswas2/Fact-Checking/Aggregated Evidence/scifact_joint_list.txt",
    sep=r"\[SEP\]",  # Use regex to handle [SEP] as the separator
    header=None,
    engine="python",
    names=["Claim", "Evidence"]
)

# Load PubMed data
pubmed_data = pd.read_csv(
    "scifact_clinicalBERT_joint_lines.txt",
    sep="\[SEP\]",
    header=None,
    names=["Claim", "Evidence"]
)

# Combine the datasets
google_data['Source'] = "Google"
pubmed_data['Source'] = "PubMed"
combined_data = pd.concat([google_data, pubmed_data], ignore_index=True)



# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

print(combined_data)


                                                  Claim  \
0     1 in 5 million in UK have abnormal PrP positiv...   
1     32% of liver transplantation programs required...   
2     40mg/day dosage of folic acid and 2mg/day dosa...   
3     76-85% of people with severe mental disorder r...   
4     A T helper 2 cell (Th2) environment impedes di...   
5     A breast cancer patient's capacity to metaboli...   
6     A country's Vaccine Alliance (GAVI) eligibilit...   
7     A deficiency of folate increases blood levels ...   
8     A diminished ovarian reserve does not solely i...   
9     A diminished ovarian reserve is a very strong ...   
10    A high microerythrocyte count protects against...   
11    A mutation in HNF4A leads to an increased risk...   
12    A single nucleotide variant the gene DGKK is s...   
13    A strong bias in the phage genome locations wh...   
14    ALDH1 expression is associated with poorer pro...   
15    AMP-activated protein kinase (AMPK) activation... 

/tmp/ipykernel_856077/3353967838.py:13: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  pubmed_data = pd.read_csv(


In [66]:
import networkx as nx
import torch
from torch_geometric.nn import GATConv
from torch.nn import Linear

# Initialize the directed graph
graph = nx.DiGraph()

# Add claims and evidence nodes with edges
for idx, row in combined_data.iterrows():
    claim = row['Claim']
    evidence = row['Evidence']
    source = row['Source']
    
    # Add the claim node if not already added
    if claim not in graph:
        graph.add_node(claim, type="claim")
    
    # Add the evidence node with its attributes
    graph.add_node(evidence, type="evidence", source=source)
    
    # Add an edge from evidence to claim
    graph.add_edge(evidence, claim, relation="is_evidence_for")

# Check the graph statistics
print(f"Graph has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")


Graph has 2070 nodes and 1384 edges.


In [ ]:
pip install torch_geometric

In [67]:
# Create mapping of node identifiers to integer indices
node_mapping = {node: idx for idx, node in enumerate(graph.nodes)}
num_nodes = len(node_mapping)

# Map edges to integer indices
edge_index = torch.tensor(
    [(node_mapping[u], node_mapping[v]) for u, v in graph.edges],
    dtype=torch.long
).t().contiguous()

# Generate random node features (128-dimensional for each node)
node_features = torch.randn((num_nodes, 128))

# Define GAT model
class GAT(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GAT, self).__init__()
        self.conv1 = GATConv(input_dim, output_dim, heads=8, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        return x

# Encode the graph
gat_model = GAT(128, 64)
encoded_nodes = gat_model(node_features, edge_index)

print("Encoded node features shape:", encoded_nodes.shape)


Encoded node features shape: torch.Size([2070, 64])


In [70]:
# Aggregate node embeddings for claims and evidence
claim_indices = [node_mapping[node] for node in graph.nodes if graph.nodes[node]['type'] == "claim"]
evidence_indices = [node_mapping[node] for node in graph.nodes if graph.nodes[node]['type'] == "evidence"]

claim_embeddings = encoded_nodes[claim_indices]
evidence_embeddings = encoded_nodes[evidence_indices]

# Aggregate evidence embeddings
aggregated_evidence = evidence_embeddings.mean(dim=0)

# Expand aggregated evidence to match claim embeddings
aggregated_evidence_expanded = aggregated_evidence.unsqueeze(0).repeat(claim_embeddings.size(0), 1)

# Combine claim and evidence embeddings
combined_embeddings = torch.cat([claim_embeddings, aggregated_evidence_expanded], dim=-1)


print(combined_embeddings.size())

torch.Size([692, 128])


In [71]:
aligned_embeddings = []
aligned_labels = []

for claim, label in zip(scifact_claims, scifact_labels):
    if claim in graph.nodes:
        claim_idx = node_mapping[claim]
        aligned_embeddings.append(combined_embeddings[claim_idx].cpu().numpy())
        aligned_labels.append(label)

aligned_embeddings = np.array(aligned_embeddings)
aligned_labels = np.array(aligned_labels)

print(f"Aligned embeddings shape: {aligned_embeddings.shape}")
print(f"Aligned labels length: {len(aligned_labels)}")


Aligned embeddings shape: (0,)
Aligned labels length: 0


In [73]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.nn import Linear

# Assuming combined_embeddings and scifact_labels are already prepared

# Step 1: Handle Mismatched Data
# Ensure consistent length by aligning embeddings and labels
min_length = min(len(combined_embeddings), len(scifact_labels))
aligned_embeddings = combined_embeddings[:min_length]
aligned_labels = torch.tensor(scifact_labels[:min_length], dtype=torch.long)

# Step 2: Split Data
# Convert data to numpy arrays for compatibility with sklearn
X_train, X_test, y_train, y_test = train_test_split(
    aligned_embeddings.detach().cpu().numpy(),
    aligned_labels.detach().cpu().numpy(),
    test_size=0.2,
    random_state=42
)

# Convert back to PyTorch tensors
X_train, X_test = map(torch.tensor, (X_train, X_test))
y_train, y_test = map(torch.tensor, (y_train, y_test))

# Step 3: Define Classifier
classifier = Linear(aligned_embeddings.size(-1), 2)  # Binary classification: Supported/Refuted
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

# Step 4: Training Loop
for epoch in range(10):
    classifier.train()
    optimizer.zero_grad()
    logits = classifier(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}: Loss = {loss.item()}")

# Step 5: Evaluation
classifier.eval()
with torch.no_grad():
    logits_test = classifier(X_test)
    y_pred = torch.argmax(logits_test, dim=-1)

# Print Classification Report
print(classification_report(y_test.numpy(), y_pred.numpy()))

# Done


Epoch 1: Loss = 0.7103146314620972
Epoch 2: Loss = 0.7094725966453552
Epoch 3: Loss = 0.7086355090141296
Epoch 4: Loss = 0.7078033089637756
Epoch 5: Loss = 0.7069761753082275
Epoch 6: Loss = 0.7061540484428406
Epoch 7: Loss = 0.7053369879722595
Epoch 8: Loss = 0.7045252919197083
Epoch 9: Loss = 0.7037187218666077
Epoch 10: Loss = 0.7029175162315369
              precision    recall  f1-score   support

           0       0.33      0.85      0.48        41
           1       0.82      0.28      0.41        98

    accuracy                           0.45       139
   macro avg       0.57      0.56      0.44       139
weighted avg       0.67      0.45      0.43       139



In [74]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import numpy as np

# Assuming scifact_claims and scifact_labels are already prepared

# Step 1: Handle Mismatched Data
min_length = min(len(scifact_claims), len(scifact_labels))
aligned_claims = scifact_claims[:min_length]
aligned_labels = scifact_labels[:min_length]

# Step 2: Tokenizer and Model Initialization
MODEL_NAME = "bert-base-uncased"  # Replace with your desired BERT model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)  # Binary classification

# Step 3: Dataset Preparation
class SciFactDataset(Dataset):
    def __init__(self, claims, labels, tokenizer, max_length=128):
        self.claims = claims
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.claims)

    def __getitem__(self, idx):
        claim = self.claims[idx]
        label = self.labels[idx]

        # Tokenize the claim
        encoding = self.tokenizer(
            claim,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

# Create datasets
train_claims, test_claims, train_labels, test_labels = train_test_split(
    aligned_claims, aligned_labels, test_size=0.2, random_state=42
)

train_dataset = SciFactDataset(train_claims, train_labels, tokenizer)
test_dataset = SciFactDataset(test_claims, test_labels, tokenizer)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Step 4: Define Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

# Training loop
for epoch in range(3):  # Number of epochs
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}: Average Loss = {avg_loss:.4f}")

# Step 5: Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Step 6: Print Classification Report
print(classification_report(all_labels, all_preds))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/home/mbiswas2/.local/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1: Average Loss = 0.6480
Epoch 2: Average Loss = 0.5860
Epoch 3: Average Loss = 0.5068
              precision    recall  f1-score   support

           0       0.11      0.02      0.03        50
           1       0.62      0.91      0.74        89

    accuracy                           0.59       139
   macro avg       0.37      0.47      0.39       139
weighted avg       0.44      0.59      0.49       139

